# Setup & Import Data Pipeline

In [1]:
# ==========================================
# CELL 1: Setup & Import Data Pipeline
# ==========================================
import os
import cv2
import torch
import numpy as np
import pandas as pd
import librosa
import torch.nn as nn
import torch.nn.functional as F
import torchaudio.transforms as AT
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models
import warnings

# Suppress audio processing warnings for cleaner logs
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# 1. GPU Check
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Using device: {device}")

# 2. Dynamic Path Discovery
def find_fakeavceleb_paths():
    search_root = "/kaggle/input"
    if not os.path.exists(search_root):
        search_root = "."
    csv_path, dataset_root = None, None
    for root, _, files in os.walk(search_root):
        if "meta_data.csv" in files:
            csv_path = os.path.join(root, "meta_data.csv")
            dataset_root = root
            break
    if csv_path is None:
        raise FileNotFoundError("Could not locate 'meta_data.csv'. Ensure FakeAVCeleb dataset is attached!")
    return csv_path, dataset_root

METADATA_CSV_PATH, KAGGLE_DATASET_ROOT = find_fakeavceleb_paths()

# 3. Helper Transforms
class SpecAugment(nn.Module):
    def __init__(self, freq_mask_param=15, time_mask_param=35):
        super().__init__()
        self.freq_mask_param = freq_mask_param
        self.time_mask_param = time_mask_param

    def forward(self, x):
        if not self.training:
            return x
        f_len, t_len = x.shape[2], x.shape[1]
        f = np.random.randint(0, self.freq_mask_param)
        f0 = np.random.randint(0, max(1, f_len - f))
        x[:, :, f0:f0+f] = 0.0
        t = np.random.randint(0, self.time_mask_param)
        t0 = np.random.randint(0, max(1, t_len - t))
        x[:, t0:t0+t, :] = 0.0
        return x

def get_visual_transforms(is_train=True):
    if is_train:
        return transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((224, 224)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    else:
        return transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

# 4. Dataset Class
class FakeAVCelebDataset(Dataset):
    def __init__(self, metadata_csv, dataset_root, seq_len=16, is_train=True):
        self.dataset_root = dataset_root
        self.seq_len = seq_len
        self.is_train = is_train
        self.df = pd.read_csv(metadata_csv)
        self.mel_transform = AT.MelSpectrogram(sample_rate=16000, n_fft=1024, hop_length=512, n_mels=128)
        self.v_transform = get_visual_transforms(is_train)
        self.spec_aug = SpecAugment() if is_train else nn.Identity()

    def __len__(self):
        return len(self.df)

    def _sample_video_frames(self, video_path):
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if not cap.isOpened() or total_frames <= 0:
            cap.release()
            return torch.zeros((self.seq_len, 3, 224, 224))
        frame_indices = np.linspace(0, total_frames - 1, self.seq_len, dtype=int)
        frames = []
        for idx in frame_indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            success, frame = cap.read()
            if success and frame is not None:
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frames.append(self.v_transform(frame_rgb))
            else:
                frames.append(torch.zeros((3, 224, 224)))
        cap.release()
        return torch.stack(frames, dim=0)

    def _extract_audio_features(self, video_path):
        try:
            y, sr = librosa.load(video_path, sr=16000, mono=True)
            waveform = torch.tensor(y, dtype=torch.float32).unsqueeze(0)
            mel_spec = self.mel_transform(waveform).squeeze(0).transpose(0, 1)
            mel_spec = mel_spec.unsqueeze(0).unsqueeze(0)
            mel_spec = F.interpolate(mel_spec, size=(self.seq_len, 128), mode='bilinear', align_corners=False).squeeze(0).squeeze(0)
            return mel_spec
        except Exception:
            return torch.zeros((self.seq_len, 128))

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        video_rel_path = str(row['path']) if 'path' in row else str(row['filename'])
        full_video_path = os.path.join(self.dataset_root, video_rel_path)
        category = str(row['type']) if 'type' in row else str(row['category'])
        label = 0.0 if "RealVideo-RealAudio" in category else 1.0

        video_tensor = self._sample_video_frames(full_video_path)
        audio_tensor = self._extract_audio_features(full_video_path)
        audio_tensor = self.spec_aug(audio_tensor.unsqueeze(0)).squeeze(0)

        return video_tensor, audio_tensor, torch.tensor(label, dtype=torch.float32)

# Load Data
full_dataset = FakeAVCelebDataset(METADATA_CSV_PATH, KAGGLE_DATASET_ROOT, seq_len=16, is_train=True)
val_size = int(0.2 * len(full_dataset))
train_size = len(full_dataset) - val_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2)
print(f"✓ Ready: {len(train_loader)} Train Batches | {len(val_loader)} Val Batches")


# ==========================================
# CELL 2: Define TSA-Net Architecture
# ==========================================
class CrossAttentionFusion(nn.Module):
    def __init__(self, embed_dim=256, num_heads=4):
        super().__init__()
        self.v_to_a_attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.a_to_v_attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.norm_v = nn.LayerNorm(embed_dim)
        self.norm_a = nn.LayerNorm(embed_dim)

    def forward(self, v_feat, a_feat):
        v_attended, _ = self.v_to_a_attn(query=v_feat, key=a_feat, value=a_feat)
        v_fused = self.norm_v(v_feat + v_attended)

        a_attended, _ = self.a_to_v_attn(query=a_feat, key=v_feat, value=v_feat)
        a_fused = self.norm_a(a_feat + a_attended)

        fused = torch.cat([v_fused, a_fused], dim=-1)
        return fused

class TSANet(nn.Module):
    def __init__(self, embed_dim=256, dropout_rate=0.3):
        super().__init__()
        effnet = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        self.v_backbone = effnet.features
        self.v_fc = nn.Linear(1280, embed_dim)

        self.a_backbone = nn.Sequential(
            nn.Conv1d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Conv1d(128, embed_dim, kernel_size=3, padding=1),
            nn.BatchNorm1d(embed_dim),
            nn.ReLU()
        )

        self.cross_attn = CrossAttentionFusion(embed_dim=embed_dim)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim * 2, 128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, 1)
        )

    def forward(self, v, a):
        B, T, C, H, W = v.shape
        v_flat = v.view(B * T, C, H, W)
        v_feat = self.v_backbone(v_flat)
        v_feat = F.adaptive_avg_pool2d(v_feat, (1, 1)).squeeze(-1).squeeze(-1)
        v_feat = self.v_fc(v_feat).view(B, T, -1)

        a_feat = a.transpose(1, 2)
        a_feat = self.a_backbone(a_feat).transpose(1, 2)

        fused = self.cross_attn(v_feat, a_feat)
        fused_pooled = fused.mean(dim=1)

        logits = self.classifier(fused_pooled).squeeze(-1)
        return logits

print("✓ TSA-Net Architecture Defined Successfully.")


# ==========================================
# CELL 3: Training & Validation Loop with Export
# ==========================================
import json
from sklearn.metrics import roc_auc_score, accuracy_score

def compute_eer(y_true, y_scores):
    from scipy.optimize import brentq
    from scipy.interpolate import interp1d
    from sklearn.metrics import roc_curve
    fpr, tpr, thresholds = roc_curve(y_true, y_scores, pos_label=1)
    eer = brentq(lambda x : 1.0 - x - interp1d(fpr, tpr)(x), 0.0, 1.0)
    return eer * 100.0

def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for v, a, labels in dataloader:
        v, a, labels = v.to(device), a.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(v, a)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)

@torch.no_grad()
def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    for v, a, labels in dataloader:
        v, a, labels = v.to(device), a.to(device), labels.to(device)
        logits = model(v, a)
        loss = criterion(logits, labels)
        total_loss += loss.item()
        
        probs = torch.sigmoid(logits).cpu().numpy()
        all_preds.extend(probs)
        all_labels.extend(labels.cpu().numpy())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    auc = roc_auc_score(all_labels, all_preds) * 100.0
    eer = compute_eer(all_labels, all_preds)
    acc = accuracy_score(all_labels, (all_preds >= 0.5).astype(int)) * 100.0

    return total_loss / len(dataloader), auc, eer, acc

model = TSANet(embed_dim=256, dropout_rate=0.3).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

NUM_EPOCHS = 5

history_tsa = {
    'train_loss': [], 'val_loss': [],
    'val_auc': [], 'val_eer': [], 'val_acc': []
}

print("Starting TSA-Net Phase 2 Training Loop...")
print("-" * 65)
print(f"{'Epoch':<8}{'Train Loss':<14}{'Val Loss':<12}{'AUC (%)':<12}{'EER (%)':<12}{'Acc (%)':<12}")
print("-" * 65)

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, auc, eer, acc = evaluate(model, val_loader, criterion, device)
    scheduler.step()

    history_tsa['train_loss'].append(train_loss)
    history_tsa['val_loss'].append(val_loss)
    history_tsa['val_auc'].append(auc)
    history_tsa['val_eer'].append(eer)
    history_tsa['val_acc'].append(acc)

    print(f"{epoch:<8}{train_loss:<14.4f}{val_loss:<12.4f}{auc:<12.2f}{eer:<12.2f}{acc:<12.2f}")

torch.save(model.state_dict(), "tsa_net_fakeavceleb_phase2.pth")
print("-" * 65)
print("✓ Training Completed! Model saved to 'tsa_net_fakeavceleb_phase2.pth'")

os.makedirs('/kaggle/working/metrics', exist_ok=True)
with open('/kaggle/working/metrics/tsa_net_phase2_history.json', 'w') as f:
    json.dump(history_tsa, f, indent=2)

print("✓ Metrics saved to '/kaggle/working/metrics/tsa_net_phase2_history.json'")

✓ Using device: cuda
✓ Ready: 2157 Train Batches | 540 Val Batches
✓ TSA-Net Architecture Defined Successfully.
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 158MB/s]


Starting TSA-Net Phase 2 Training Loop...
-----------------------------------------------------------------
Epoch   Train Loss    Val Loss    AUC (%)     EER (%)     Acc (%)     
-----------------------------------------------------------------
1       0.1182        0.1151      50.01       49.99       97.87       
2       0.1163        0.1033      50.00       50.00       97.87       
3       0.1163        0.1031      50.00       50.00       97.87       
4       0.1161        0.1031      49.99       50.01       97.87       
5       0.1134        0.1033      50.01       49.99       97.87       
-----------------------------------------------------------------
✓ Training Completed! Model saved to 'tsa_net_fakeavceleb_phase2.pth'
✓ Metrics saved to '/kaggle/working/metrics/tsa_net_phase2_history.json'
